## Key Learning Objectives:
1. Apply appropriate feature scaling (`StandardScaler`, `RobustScaler`) to continuous variables.
2. Reduce feature skewness using logarithmic transformations (`np.log1p`).
3. Encode nominal and ordinal categorical variables correctly.
4. Filter out low-variance and collinear features.
5. Rank and select top features using statistical tests (`SelectKBest`).

In [3]:
# Setup & Dataset Generation

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# Seed for reproducibility
np.random.seed(2026)
n_samples = 300

# Synthetic Financial Credit Scoring Dataset
age = np.random.normal(loc=42, scale=12, size=n_samples).round()
annual_income = np.random.exponential(scale=60000, size=n_samples) + 20000  # Skewed
credit_score = np.random.uniform(300, 850, size=n_samples)
loan_amount = annual_income * np.random.uniform(0.2, 0.8, size=n_samples) # Strongly correlated with annual_income

# Categorical variables
education_level = np.random.choice(["High School", "Bachelor", "Master", "PhD"], size=n_samples, p=[0.2, 0.5, 0.2, 0.1])
home_ownership = np.random.choice(["RENT", "MORTGAGE", "OWN"], size=n_samples, p=[0.4, 0.4, 0.2])

# Constant / Quasi-constant features
country_code = [91] * n_samples  # Zero variance
has_rare_bankruptcy = [1] + [0] * (n_samples - 1)  # Quasi-constant

# Target: Default likelihood
default_prob = 1 / (1 + np.exp(-((700 - credit_score) / 100 + (loan_amount / annual_income))))
loan_default = np.random.binomial(1, default_prob)

df_credit = pd.DataFrame({
    "age": age,
    "annual_income": annual_income.round(2),
    "credit_score": credit_score.round(1),
    "loan_amount": loan_amount.round(2),
    "education_level": education_level,
    "home_ownership": home_ownership,
    "country_code": country_code,
    "has_rare_bankruptcy": has_rare_bankruptcy,
    "loan_default": loan_default
})

# Inject extreme income outliers
df_credit.loc[[5, 25, 80], "annual_income"] = [1200000.0, 1500000.0, 2000000.0]

print("--- Credit Dataset Sample ---")
print(df_credit.head(6))

--- Credit Dataset Sample ---
    age  annual_income  credit_score  loan_amount education_level  \
0  37.0       62654.30         517.4     37697.20        Bachelor   
1  25.0       36173.41         529.3     17137.92        Bachelor   
2  46.0       43092.33         462.1      9394.24     High School   
3  42.0      114589.55         524.4     30700.62             PhD   
4  59.0      208167.25         431.9    121397.32        Bachelor   
5  46.0     1200000.00         561.7     43829.31        Bachelor   

  home_ownership  country_code  has_rare_bankruptcy  loan_default  
0       MORTGAGE            91                    1             1  
1           RENT            91                    0             1  
2           RENT            91                    0             1  
3       MORTGAGE            91                    0             0  
4            OWN            91                    0             1  
5            OWN            91                    0             1  


---
## Task 1: Handling Skewness & Scaling Continuous Features

**Instructions:**
1. Compute the skewness of `annual_income`. Apply `np.log1p()` to create a new column `log_annual_income` to fix right-skewness.
2. Apply `RobustScaler` to `log_annual_income` and `StandardScaler` to `age` and `credit_score`.

In [4]:
# Task 1 Prompt
# TODO: Calculate skewness, apply log transform, and scale continuous numerical features
# Hint: Use np.log1p() and sklearn.preprocessing scalers
from sklearn.preprocessing import RobustScaler, StandardScaler
# YOUR CODE HERE
income_skewness = df_credit["annual_income"].skew()

print("Skewness before transformation:", income_skewness)

df_credit["log_annual_income"] = np.log1p(df_credit["annual_income"])

log_income_skewness = df_credit["log_annual_income"].skew()

print("Skewness after transformation:", log_income_skewness)

robust_scaler = RobustScaler()

df_credit["log_income_scaled"] = robust_scaler.fit_transform(
    df_credit[["log_annual_income"]]
)

standard_scaler = StandardScaler()

df_credit[["age_scaled", "credit_score_scaled"]] = standard_scaler.fit_transform(
    df_credit[["age", "credit_score"]]
)

print("\nScaled Features:")
print(
    df_credit[
        [
            "annual_income",
            "log_annual_income",
            "log_income_scaled",
            "age",
            "age_scaled",
            "credit_score",
            "credit_score_scaled"
        ]
    ].head()
)

Skewness before transformation: 8.632257411814576
Skewness after transformation: 0.8120233414140957

Scaled Features:
   annual_income  log_annual_income  log_income_scaled   age  age_scaled  \
0       62654.30          11.045404          -0.043962  37.0   -0.476521   
1       36173.41          10.496107          -0.591242  25.0   -1.465268   
2       43092.33          10.671124          -0.416868  46.0    0.265039   
3      114589.55          11.649121           0.557538  42.0   -0.064543   
4      208167.25          12.246102           1.152327  59.0    1.336182   

   credit_score  credit_score_scaled  
0         517.4            -0.418602  
1         529.3            -0.340695  
2         462.1            -0.780637  
3         524.4            -0.372774  
4         431.9            -0.978349  


---
## Task 2: Categorical Encoding (Nominal & Ordinal)

**Instructions:**
1. Encode `education_level` using `OrdinalEncoder` with the explicit hierarchy:
   `["High School", "Bachelor", "Master", "PhD"]`.
2. Encode `home_ownership` using `OneHotEncoder` with `drop="first"` and `sparse_output=False`.

In [5]:
# Task 2 Prompt
# TODO: Ordinal encode education_level and One-Hot encode home_ownership
ordinal_mapping = OrdinalEncoder(
    categories=[["High School", "Bachelor", "Master", "PhD"]]
)
df_credit["education_level_ecoded"] = ordinal_mapping.fit_transform(df_credit[["education_level"]])
# YOUR CODE HERE

one_hot = OneHotEncoder(
    drop="first",
    sparse_output=False
)

encoded_data = one_hot.fit_transform(df_credit[["home_ownership"]])

encoded_columns = one_hot.get_feature_names_out(["home_ownership"])

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoded_columns,
    index=df_credit.index
)

df_credit = pd.concat([df_credit, encoded_df], axis=1)
print(df_credit.head())

    age  annual_income  credit_score  loan_amount education_level  \
0  37.0       62654.30         517.4     37697.20        Bachelor   
1  25.0       36173.41         529.3     17137.92        Bachelor   
2  46.0       43092.33         462.1      9394.24     High School   
3  42.0      114589.55         524.4     30700.62             PhD   
4  59.0      208167.25         431.9    121397.32        Bachelor   

  home_ownership  country_code  has_rare_bankruptcy  loan_default  \
0       MORTGAGE            91                    1             1   
1           RENT            91                    0             1   
2           RENT            91                    0             1   
3       MORTGAGE            91                    0             0   
4            OWN            91                    0             1   

   log_annual_income  log_income_scaled  age_scaled  credit_score_scaled  \
0          11.045404          -0.043962   -0.476521            -0.418602   
1          10.4961

---
## Task 3: Low-Variance Feature Filtering

**Instructions:**
1. Use `VarianceThreshold(threshold=0.01)` to identify and remove constant or quasi-constant numerical features (`country_code`, `has_rare_bankruptcy`).
2. Print the list of dropped features and retained features.

In [6]:
# Task 3 Prompt
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
# TODO: Apply VarianceThreshold to drop low-variance columns
variance_threshold = VarianceThreshold(threshold=0.01)
X_nums = df_credit.select_dtypes(include=[np.number])
X_filtered = variance_threshold.fit_transform(X_nums)

# 4. Identify retained and dropped columns dynamically
retained_features = X_nums.columns[variance_threshold.get_support()].to_list()

dropped_features = X_nums.columns[~variance_threshold.get_support()].to_list()
# YOUR CODE HERE
print("Dropped Features:", dropped_features)
print("Retained Features:", retained_features)

df_credit_filtered = df_credit.drop(columns=dropped_features)


Dropped Features: ['country_code', 'has_rare_bankruptcy']
Retained Features: ['age', 'annual_income', 'credit_score', 'loan_amount', 'loan_default', 'log_annual_income', 'log_income_scaled', 'age_scaled', 'credit_score_scaled', 'education_level_ecoded', 'home_ownership_OWN', 'home_ownership_RENT']


---
## Task 4: Removing Highly Collinear Features

**Instructions:**
1. Calculate the Pearson correlation matrix for numerical features.
2. Identify features with absolute correlation $|r| > 0.85$ (e.g., `annual_income` vs `loan_amount`) and drop the redundant column.

In [7]:
# Task 4 Prompt

# TODO: Detect and drop collinear features with |r| > 0.85
X_nums = df_credit.select_dtypes(include=[np.number])

corr_matrix = X_nums.corr().abs()

upper_trainge = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
threshold = 0.85
dropped_features = [column for column in upper_trainge.columns if any(upper_trainge[column] > threshold)]
# YOUR CODE HERE
print("Highly correlated features to drop:", dropped_features)

# 6. Drop correlated features
df_credit_filtered = df_credit.drop(columns=dropped_features)

print("\nRemaining Features:")
print(df_credit_filtered.columns.tolist())

Highly correlated features to drop: ['log_income_scaled', 'age_scaled', 'credit_score_scaled']

Remaining Features:
['age', 'annual_income', 'credit_score', 'loan_amount', 'education_level', 'home_ownership', 'country_code', 'has_rare_bankruptcy', 'loan_default', 'log_annual_income', 'education_level_ecoded', 'home_ownership_OWN', 'home_ownership_RENT']


---
## Task 11: Statistical Feature Selection (`SelectKBest`)

**Instructions:**
1. Rank all remaining numerical features against the target `loan_default` using `SelectKBest` with `f_classif` (ANOVA F-Test) and `mutual_info_classif`.
2. Plot a comparative bar chart showing feature importance scores.
3. *Note:* Remember to specify `hue` and set `legend=False` in Seaborn barplots to prevent deprecation warnings.

In [ ]:
# Task 5 Prompt

# TODO: Rank features using SelectKBest (f_classif and mutual_info_classif) and plot the scores

# YOUR CODE HERE